# Data Visualization for Data Engineers — When Should I Use Each Plot?

## Learning objective

By the end of this notebook, you should be able to look at a data question and decide:

- **Line plot** → Do I want to see change over an ordered sequence, especially time?
- **Bar plot** → Do I want to compare categories?
- **Histogram** → Do I want to understand the distribution of one numerical variable?
- **Scatter plot** → Do I want to examine the relationship between two numerical variables?
- **Boxplot** → Do I want to compare distributions and identify potential outliers?

The goal is **not** to memorize plotting syntax. The goal is to learn the reasoning behind choosing a visualization.

> Data Engineer mindset: visualization is often used for data exploration, validation, quality checks, debugging, and communicating what is happening in a dataset.

# 1. Setup

We will use a small synthetic dataset inspired by two areas relevant to you:

1. **Data Engineering / job data**
2. **Agriculture and irrigation data**

We will use pandas and matplotlib.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (9, 5)

# Reproducible random data
np.random.seed(42)

dates = pd.date_range("2026-01-01", periods=30, freq="D")

df = pd.DataFrame({
    "date": dates,
    "records_processed": np.random.randint(8000, 16000, 30),
    "processing_time_min": np.random.normal(25, 5, 30),
    "salary": np.random.normal(75000, 18000, 100),
    "experience_years": np.random.uniform(0, 12, 100),
    "country": np.random.choice(
        ["United States", "Canada", "United Kingdom", "France", "Germany"],
        100
    ),
    "irrigation_mm": np.random.uniform(10, 100, 100),
})

df["yield_t_ha"] = 2 + 0.035 * df["irrigation_mm"] + np.random.normal(0, 0.5, 100)

df.head()

# 2. The first question: what am I trying to see?

Before writing `plt...`, ask:

### Question A — Is there an ordered variable such as time?
Use a **line plot** when you want to see a trend or change over an ordered sequence.

### Question B — Am I comparing separate categories?
Use a **bar plot**.

### Question C — Am I studying one numerical variable?
Use a **histogram** to see its distribution.

### Question D — Am I studying two numerical variables together?
Use a **scatter plot**.

### Question E — Do I want to compare distributions or inspect potential outliers?
Use a **boxplot**.

This decision process is more important than memorizing the commands.

# 3. LINE PLOT — When do I use it?

## Main rule

Use a **line plot when the order of the x-axis matters**.

The most common case is **time**.

### Good question

> How did the number of records processed by my pipeline change during the last 30 days?

That is a line-plot question because the dates have a natural order.

In [ ]:
plt.plot(df["date"], df["records_processed"])

plt.title("Records processed by day")
plt.xlabel("Date")
plt.ylabel("Number of records")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Data Engineer examples

A line plot can help you inspect:

- records processed per day
- pipeline execution time
- API requests over time
- daily data volume
- storage growth
- CPU or memory usage over time

### Decision rule

**"How does X change over time?" → Line plot**

### Important distinction

Do not automatically use a line plot just because you have numbers.

If you are comparing countries such as USA, Canada, France and Germany, those categories do not form a continuous sequence. A **bar plot** is usually more appropriate.

# 4. BAR PLOT — When do I use it?

## Main rule

Use a **bar plot to compare categories**.

Example question:

> Which countries have the most job postings?

Here, the countries are categories.

In [ ]:
country_counts = df["country"].value_counts()

country_counts.plot(kind="bar")

plt.title("Number of observations by country")
plt.xlabel("Country")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Data Engineer examples

Bar plots are useful for:

- number of records by country
- number of jobs by job title
- number of errors by error type
- number of customers by region
- number of records by source system

### Decision rule

**"I want to compare categories." → Bar plot**

### Bar plot vs line plot

Suppose you have:

`USA, Canada, France, Germany`

→ **Bar plot**

Suppose you have:

`Monday, Tuesday, Wednesday, Thursday`

and you want to see how a metric changes over those days:

→ **Line plot**

# 5. HISTOGRAM — When do I use it?

## Main rule

Use a **histogram to understand the distribution of one numerical variable**.

Question:

> How are salaries distributed in this dataset?

We don't necessarily care about individual salaries. We want to understand the overall distribution.

In [ ]:
plt.hist(df["salary"], bins=15)

plt.title("Distribution of salary")
plt.xlabel("Salary")
plt.ylabel("Frequency")
plt.show()

### What can a histogram tell you?

You can inspect whether values appear:

- concentrated in one range
- spread out
- skewed
- unusually distributed
- potentially affected by extreme values

### Data Engineer examples

Use histograms when exploring:

- transaction amounts
- salaries
- processing times
- file sizes
- response times
- sensor measurements

### Decision rule

**"How are the values distributed?" → Histogram**

A histogram is especially useful during **exploratory data analysis and data-quality investigation**.

# 6. SCATTER PLOT — When do I use it?

## Main rule

Use a **scatter plot when you want to examine the relationship between two numerical variables**.

Agriculture example:

> Is there a relationship between irrigation amount and crop yield?

In [ ]:
plt.scatter(df["irrigation_mm"], df["yield_t_ha"])

plt.title("Irrigation amount vs crop yield")
plt.xlabel("Irrigation (mm)")
plt.ylabel("Yield (t/ha)")
plt.show()

Each point represents one observation.

Here:

- X = irrigation amount
- Y = crop yield

### Data Engineer examples

Scatter plots can help explore:

- experience vs salary
- processing time vs file size
- CPU usage vs response time
- advertising spend vs revenue
- irrigation vs yield

### Decision rule

**"Do these two numerical variables have a relationship?" → Scatter plot**

Important: a scatter plot can show an association, but it does **not automatically prove causation**.

# 7. BOXPLOT — When do I use it?

## Main rule

Use a **boxplot to summarize a numerical distribution and compare distributions between groups**.

It is also useful for spotting **potential outliers**.

In [ ]:
plt.boxplot(df["salary"], vert=True)

plt.title("Salary distribution")
plt.ylabel("Salary")
plt.show()

A boxplot gives a compact view of the distribution using the median, quartiles and potential outlying observations.

### Compare groups

For example, compare salary distributions by country.

In [ ]:
groups = [
    df.loc[df["country"] == country, "salary"]
    for country in df["country"].unique()
]

plt.boxplot(groups, labels=df["country"].unique())

plt.title("Salary distribution by country")
plt.xlabel("Country")
plt.ylabel("Salary")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Data Engineer examples

Boxplots can help investigate:

- processing time by data source
- transaction values by customer segment
- salary by country
- sensor measurements by location
- data-volume differences between systems

### Decision rule

**"How do distributions differ between groups, and are there potential outliers?" → Boxplot**

# 8. The five plots side by side conceptually

| Plot | Main question | Typical Data Engineering use |
|---|---|---|
| Line | How does something change over an ordered sequence? | Pipeline monitoring |
| Bar | How do categories compare? | Counts by country/source/type |
| Histogram | How are numerical values distributed? | Data exploration/quality |
| Scatter | How are two numerical variables related? | Relationship investigation |
| Boxplot | How do distributions compare? Are there potential outliers? | Data-quality checks |

## The key idea

Do not start with:

> "Which plot do I know how to code?"

Start with:

> **"What question am I asking about my data?"**

# 9. A practical Data Engineer example

Imagine you receive a daily dataset from several systems.

You have:

- `date`
- `source`
- `records`
- `processing_time`
- `file_size`
- `error_count`

Which plot would you choose?

### Situation 1

> I want to know whether the number of records processed is increasing over time.

**Answer: Line plot**

### Situation 2

> I want to compare how many records came from each source.

**Answer: Bar plot**

### Situation 3

> I want to understand the distribution of processing times.

**Answer: Histogram**

### Situation 4

> I want to see whether larger files take longer to process.

**Answer: Scatter plot**

### Situation 5

> I want to compare processing-time distributions between sources and identify potential unusual values.

**Answer: Boxplot**

# 10. Practice — Choose the plot before coding

For each question, write one of:

`line`, `bar`, `histogram`, `scatter`, `boxplot`

### Exercise 1
You want to compare the number of jobs posted in the USA, UK, Canada and France.

### Exercise 2
You want to see how daily pipeline execution time changes over six months.

### Exercise 3
You want to understand the distribution of salaries.

### Exercise 4
You want to investigate whether years of experience are associated with salary.

### Exercise 5
You want to compare salary distributions across several countries and inspect potential outliers.

### Exercise 6 — Agriculture
You want to investigate the relationship between irrigation amount and crop yield.

### Exercise 7 — Data Quality
You want to inspect whether processing times contain unusually high observations.

# 11. Practice — Now code it yourself

Use the dataset above.

### Task 1 — Bar plot
Show the number of observations for each country.

### Task 2 — Histogram
Create a histogram of `processing_time_min`.

### Task 3 — Scatter plot
Plot `experience_years` against `salary`.

### Task 4 — Line plot
Plot `records_processed` against `date`.

### Task 5 — Boxplot
Create a boxplot of `processing_time_min`.

For each task, write a comment answering:

> Why did I choose this plot?

# 12. A simple decision tree to memorize

Ask these questions in order:

**1. Is my x-variable time or another ordered sequence?**

→ Yes: **Line plot**

**2. Am I comparing categories?**

→ Yes: **Bar plot**

**3. Am I studying the distribution of one numerical variable?**

→ Yes: **Histogram**

**4. Am I studying the relationship between two numerical variables?**

→ Yes: **Scatter plot**

**5. Am I comparing distributions or checking potential outliers?**

→ Yes: **Boxplot**

## One important reminder

These rules are guidelines, not laws. The right visualization ultimately depends on the question, the data structure, and what you want the reader to understand.

# 13. Data Engineer priority

For your current Data Engineering learning path:

### Learn well
- pandas
- filtering and grouping
- aggregation
- data cleaning
- basic Matplotlib
- choosing the correct visualization
- interpreting simple charts

### Do not over-invest yet
You do not need to become a specialist in advanced visualization.

Your larger priority should be:

**Python → Pandas → SQL → databases → ETL/ELT → APIs → Linux → Git → Docker → orchestration → cloud**

Visualization is a supporting skill that helps you **understand, validate, debug and communicate data**.